# Data and split validation
COM747 DS&ML CW2 — Multidimensional care-intensity discordance.

Run after `cleaning.ipynb` using **Python (DSML CW2)**. This notebook independently checks exported data, lineage, cohort exclusions, feature contract and patient-level separation. It does not train models, select thresholds or assess model performance. It uses held-out labels only for integrity checks, not exploratory results. A passing audit is necessary but not sufficient for clinical validity. Detailed fairness and model evaluation remain later steps.

In [1]:
from pathlib import Path
import hashlib, json, itertools
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display
import sys

root = next((p for p in [Path.cwd(), Path.cwd().parent] if (p/"data/processed/feature_contract.json").is_file()), None)
if root is None:
    raise FileNotFoundError("Run cleaning.ipynb first from the project root or cleaning/.")
OUT = root / "data/processed"
RAW = root / "data/raw/diabetic_data.csv"
contract = json.loads((OUT/"feature_contract.json").read_text(encoding="utf-8"))

def digest(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest() if hasattr(hashlib, "file_digest") else hashlib.sha256(f.read()).hexdigest()

def load(path):
    return pd.read_csv(path, keep_default_na=False, na_values=["?", ""], dtype={"encounter_id":"string", "patient_nbr":"string"})

checks = []
def check(name, passed, detail=""):
    checks.append({"check":name, "status":"PASS" if bool(passed) else "FAIL", "detail":detail})

# Verify integrity before trusting the exported artefacts.
check("Raw source checksum", RAW.is_file() and digest(RAW) == contract["source_sha256"])
map_path = next((p for p in [RAW.parent/"IDS_mapping.csv", RAW.parent/"IDs_mapping.csv"] if p.is_file()), None)
if contract["mapping_verified"]:
    check("Mapping checksum", map_path is not None and digest(map_path) == contract["mapping_sha256"])
for name, expected in contract["output_sha256"].items():
    p = OUT/name
    check(f"Checksum: {name}", p.is_file() and digest(p) == expected)
if any(c["status"] == "FAIL" for c in checks):
    display(pd.DataFrame(checks))
    raise ValueError("Source/output files changed or are absent; resolve integrity checks first.")
print("Integrity checks passed. Kernel:", sys.executable)

Integrity checks passed. Kernel: /Users/monishachandrashekar/Desktop/DS&ML-CW2 /.venv/bin/python


## 1. Validate independent partitions and feature schema
No patient or encounter may appear in more than one partition. Validate feature whitelists rather than dropping a few forbidden columns, because new accidental columns could leak outcomes. Verify that numerical care counts are unchanged in meaning, that nominal admission IDs remain nominal at modelling time, and that each split contains both classes. Do not display held-out label rates or patient identifiers.

In [2]:
IDS = [contract["id"], contract["group"]]
TARGET = contract["target"]
CARE = contract["observed_care_features"]
CLINICAL = contract["clinical_features"]
expected_cols = IDS + CLINICAL + CARE + contract["audit_only"] + [TARGET, "split"]
parts = {name:load(OUT/f"{name}.csv") for name in ["train","validation","test"]}
for name, part in parts.items():
    check(f"{name}: exact schema", list(part.columns) == expected_cols)
    check(f"{name}: correct split", part["split"].eq(name).all())
    check(f"{name}: valid IDs", part[IDS].notna().all().all() and part[IDS[0]].is_unique)
    check(f"{name}: binary target with both classes", part[TARGET].notna().all() and set(part[TARGET]) == {0,1})
    care = part[CARE].apply(pd.to_numeric, errors="coerce")
    check(f"{name}: care counts valid", care.notna().all().all() and (care >= 0).all().all() and (care % 1 == 0).all().all() and care["time_in_hospital"].between(1,14).all())
    for col in contract["clinical_numeric"]:
        nums = pd.to_numeric(part[col], errors="coerce")
        check(f"{name}: {col} valid or missing", not (part[col].notna() & nums.isna()).any() and (nums.dropna() >= 0).all() and (nums.dropna() % 1 == 0).all())
    check(f"{name}: eligible discharge disposition", part["discharge_disposition_id"].notna().all() and not part["discharge_disposition_id"].isin(contract["excluded_dispositions"]).any())

for a,b in itertools.combinations(parts,2):
    for col in IDS:
        check(f"{a}/{b}: no overlap in {col}", set(parts[a][col]).isdisjoint(parts[b][col]))

all_data = pd.concat(parts.values(), ignore_index=True)
check("Unique encounters across cohort", all_data[IDS[0]].is_unique)
check("Expected cohort size", len(all_data) == contract["eligible_rows"])
assign = load(OUT/"split_assignments.csv")
expected_assign = all_data[IDS+["split"]].sort_values(IDS[0]).reset_index(drop=True)
check("Split assignment file matches partitions", assign.sort_values(IDS[0]).reset_index(drop=True).equals(expected_assign))
forbidden = set(IDS + [TARGET, "readmitted", "split"] + contract["audit_only"])
check("Set A exact whitelist", contract["feature_set_A"] == CLINICAL and not forbidden.intersection(CLINICAL))
check("Set B exact whitelist", contract["feature_set_B"] == CLINICAL+CARE and not forbidden.intersection(contract["feature_set_B"]))
check("Set C planned whitelist", contract["feature_set_C_planned"] == CLINICAL+CARE+contract["planned_discordance_features"] and not forbidden.intersection(contract["feature_set_C_planned"]))
check("Numeric/categorical contract", set(contract["clinical_numeric"]).isdisjoint(contract["clinical_categorical"]) and set(contract["clinical_numeric"]+contract["clinical_categorical"]) == set(CLINICAL))
check("No fitted discordance features in prepared data", not set(contract["planned_discordance_features"]).intersection(all_data.columns))

## 2. Verify labels, original care counts and cohort lineage against raw source
Recreate eligibility from documented fixed rules. Confirm the binary target, patient ID, four care counts and literal `None` laboratory values survive the cleaning export. These are deterministic checks, not statistical hypothesis tests. Exact duplicate rows may legitimately be removed; repeated encounters for different admission records are not duplicates.

In [3]:
source = pd.read_csv(RAW, keep_default_na=False, na_values=["?", ""], dtype={"encounter_id":"string","patient_nbr":"string","diag_1":"string","diag_2":"string","diag_3":"string"})
check("Raw row count", len(source) == contract["raw_rows"])
for col in source.select_dtypes(include=["object","string"]).columns:
    source[col] = source[col].astype("string").str.strip().replace({"?":pd.NA,"":pd.NA})
source = source.drop_duplicates()
for col in CARE + ["discharge_disposition_id"]:
    source[col] = pd.to_numeric(source[col], errors="coerce")
valid = (source["discharge_disposition_id"].notna() & ~source["discharge_disposition_id"].isin(contract["excluded_dispositions"]) & source[CARE].notna().all(axis=1) & source["time_in_hospital"].between(1,14))
source = source.loc[valid].set_index(IDS[0])
exported = all_data.set_index(IDS[0])
check("Eligible encounter set equals raw-derived cohort", set(source.index) == set(exported.index) and source.index.is_unique)
if source.index.is_unique and exported.index.is_unique and set(source.index) == set(exported.index):
    source = source.reindex(exported.index)
    check("Patient IDs preserved", source[IDS[1]].astype("string").equals(exported[IDS[1]].astype("string")))
    check("Binary target matches raw label", np.array_equal(source["readmitted"].eq("<30").astype(int).to_numpy(), exported[TARGET].to_numpy()))
    for col in CARE:
        check(f"Observed care preserved: {col}", np.array_equal(source[col].to_numpy(dtype=float), exported[col].to_numpy(dtype=float)))
    for col in ["A1Cresult", "max_glu_serum"]:
        check(f"Lab test category preserved: {col}", source[col].astype("string").equals(exported[col].astype("string")))
else:
    check("Raw-to-export value checks", False, "Encounter sets or uniqueness mismatch")
flow = pd.read_csv(OUT/"cohort_flow.csv")
check("Cohort flow reconciles", flow.iloc[0]["remaining"] == contract["raw_rows"] and flow.iloc[-1]["remaining"] == len(all_data) and (flow["removed"].iloc[1:].to_numpy() == -flow["remaining"].diff().iloc[1:].to_numpy()).all())

/var/folders/2m/b7dfp3r133v8frt6g9kbg0v80000gn/T/ipykernel_6194/3780651388.py:1: DtypeWarning: Columns (10: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  source = pd.read_csv(RAW, keep_default_na=False, na_values=["?", ""], dtype={"encounter_id":"string","patient_nbr":"string","diag_1":"string","diag_2":"string","diag_3":"string"})


## 3. Training-only demographic coverage and missingness audit
Examine the training cohort to identify groups that may need cautious interpretation. Counts below a prespecified minimum of 20 encounters or 20 unique patients are suppressed in this display; this is a conservative reporting choice, not a guarantee of anonymity or a statistical power calculation. Report missing categories explicitly. Later fairness evaluation should also require sufficient positive and negative cases, use patient-cluster bootstrap confidence intervals, and never treat a small gap as proof of equity.

In [4]:
train = parts["train"]
MIN_GROUP = 20
for col in ["age","gender","race"]:
    report = train.assign(group=train[col].astype("string").fillna("Missing")).groupby("group").agg(encounters=(IDS[0],"size"), patients=(IDS[1],"nunique"), positives=(TARGET,"sum"))
    report["readmission_rate"] = report["positives"]/report["encounters"]
    small = (report["encounters"] < MIN_GROUP) | (report["patients"] < MIN_GROUP) | (report["positives"] < MIN_GROUP) | ((report["encounters"]-report["positives"]) < MIN_GROUP)
    safe = report.astype(object).mask(small, "Suppressed", axis=0)
    print("Training coverage:", col)
    display(safe)
print("Training missingness audit saved by cleaning in training_missingness.csv.")

Training coverage: age


,encounters,patients,positives,readmission_rate
group,,,,
[0-10),Suppressed,Suppressed,Suppressed,Suppressed
[10-20),397,305,25,0.062972
[20-30),1016,688,155,0.152559
[30-40),2296,1666,248,0.108014
[40-50),5612,4065,583,0.103885
[50-60),10186,7525,1008,0.098959
[60-70),13469,9782,1533,0.113817
[70-80),15225,10821,1831,0.120263
[80-90),9822,6872,1241,0.126349


Training coverage: gender


,encounters,patients,positives,readmission_rate
group,,,,
Female,32044,22382,3644,0.113719
Male,27558,19649,3144,0.114087
Unknown/Invalid,Suppressed,Suppressed,Suppressed,Suppressed


Training coverage: race


,encounters,patients,positives,readmission_rate
group,,,,
AfricanAmerican,11118,7522,1270,0.114229
Asian,356,297,40,0.11236
Caucasian,44595,31477,5164,0.115798
Hispanic,1264,952,122,0.096519
Missing,1392,1206,120,0.086207
Other,879,718,72,0.081911


Training missingness audit saved by cleaning in training_missingness.csv.


## 4. Save validation evidence and stop if any check fails
The JSON audit contains only aggregate checks and configuration flags, not patient identifiers. The mapping warning must be resolved before the final coursework report. A successful validation report does not establish that the dataset is representative or the model safe. The test set remains sealed for final model evaluation.

In [5]:
status = pd.DataFrame(checks)
display(status)
passed = bool(status["status"].eq("PASS").all())
report = {"timestamp_utc":datetime.now(timezone.utc).isoformat(), "passed":passed,
          "checks":checks, "mapping_verified":contract["mapping_verified"],
          "warnings": ([] if contract["mapping_verified"] else ["Official mapping was not available; verify discharge codes before submission."]),
          "scope":"Data/split integrity only; no model validation performed."}
(OUT/"validation_report.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
if not passed:
    raise AssertionError("Data validation failed. Fix the source/preparation issue before modelling.")
print(f"PASS: {len(checks)} checks. Validation report written.")
if not contract["mapping_verified"]:
    print("WARNING: obtain official IDS_mapping.csv and rerun before final submission.")

,check,status,detail
0,Raw source checksum,PASS,
1,Mapping checksum,PASS,
2,Checksum: train.csv,PASS,
3,Checksum: validation.csv,PASS,
4,Checksum: test.csv,PASS,
...,...,...,...
58,Observed care preserved: num_procedures,PASS,
59,Observed care preserved: num_medications,PASS,
60,Lab test category preserved: A1Cresult,PASS,
61,Lab test category preserved: max_glu_serum,PASS,


PASS: 63 checks. Validation report written.


## 5. Model validation protocol (next stage; not executed here)
- Expected-care regressors: patient-grouped out-of-fold predictions; MAE, RMSE and residual diagnostics per care dimension. Standardise signed residuals with training-only mean/SD, handle zero SD explicitly. No use of readmission as a predictor for expected care.
- Compare feature sets A/B/C across the same classifier configurations and partitions. For any CV-based tuning, regenerate residuals with nested group-aware cross-fitting inside each outer training fold. Alternatively tune on the fixed validation split with training-only OOF features. Do not train on validation and then compare with stale thresholds or residual scaling.
- Choose hyperparameters and classification threshold on validation only. Use average precision (report as AP, a specific PR summary), recall, precision, F1, FNR, confusion matrix, ROC-AUC and calibration on the untouched test set; AP is threshold-independent. Compare paired AP and recall differences using patient-level bootstrap; disclose class prevalence and uncertainty. Use cluster-robust inference for RQ1, avoiding simultaneous inclusion of highly redundant raw and residual terms without checking collinearity.
- Fairness: audit age, recorded gender and race, plus missingness; report subgroup sample sizes, positive counts, recall/FNR and precision when support allows. Compare sensitivity without race/gender predictors. Avoid claiming fairness based on parity alone.
- Sustainability: record elapsed training time, hardware and tuning trial counts; avoid converting time to carbon without measured energy and appropriate emissions factor. No actual healthcare costs are in the dataset.
- See `LSEP_Sustainability.md` at project root for the governance plan and evidence checklist.